In [ ]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# Resolve paths from the repository root or this notebook's folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'playground-series-s6e9' / 'train.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Download the competition CSVs into playground-series-s6e9/.')
DATA_DIR = PROJECT_ROOT / 'playground-series-s6e9'
OUTPUT_DIR = PROJECT_ROOT / 'XGBoost/baseline'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [ ]:
import xgboost as xgb

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'scale_pos_weight': float(np.sum(y == 0) / np.sum(y == 1)),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(params, dtrain, num_boost_round=1000, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        preds = model.predict(dval)
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['tree_method'] = 'hist'
best_params['scale_pos_weight'] = float(np.sum(y == 0) / np.sum(y == 1))

with open(OUTPUT_DIR / 'xgboost_best_params.json', 'w') as f:
    json.dump(best_params, f)


In [ ]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)
    
    model = xgb.train(best_params, dtrain, num_boost_round=2000, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    
    val_preds = model.predict(dval)
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict(dtest)
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv(OUTPUT_DIR / 'xgboost_oof_predictions.csv', index=False)
fold_preds_df.to_csv(OUTPUT_DIR / 'xgboost_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv(OUTPUT_DIR / 'submission_xgboost.csv', index=False)
